In [1]:
import sys
sys.path.append('/home/jwinchel/Heimdall/')

from Heimdall.utils import convert_to_ensembl_ids


In [2]:
import scanpy as sc

# read single files from ST and scRNAseq to get gene names
st_data = sc.read_csv('/compute/oven-0-13/jwinchel/Lung5_Rep1/Lung5_Rep1-Flat_files_and_images/Lung5_Rep1_exprMat_file.csv')
sc_data = sc.read_csv('/compute/oven-0-13/jwinchel/scRNA-seq/GSM3635278_human_p1t1_raw_counts.tsv', delimiter='\t')

/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:

In [3]:
st_data = st_data[:10]
st_data.obs[['fov', 'cell_ID']] = st_data.X[:,:2]
# st_data.X = st_data.X[:,2:]
st_data = st_data[:,~st_data.var_names.isin(['fov','cell_ID'])]


/tmp/ipykernel_3538962/1737458199.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_data.obs[['fov', 'cell_ID']] = st_data.X[:,:2]


In [4]:
st_data, st_gene_mapping = convert_to_ensembl_ids(st_data,'/compute/oven-0-13/jwinchel/')

human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/Heimdall/Heimdall/utils.py:457: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["gene_mapping:symbol_to_ensembl"] = gene_mapping.mapping_full


In [8]:
import pickle
# Pickling (saving to a file)
with open('gene_mapping.pickle', 'wb') as f:
    # Use the highest protocol for best performance and features
    pickle.dump(st_gene_mapping, f, protocol=pickle.HIGHEST_PROTOCOL)



In [6]:
!pwd

/home/jwinchel/Syndri/nsclc_playground


In [ ]:
sc_data = sc_data[:10]
sc_data, sc_gene_mapping = convert_to_ensembl_ids(sc_data,'/compute/oven-0-13/jwinchel/')

human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 28,057 out of 41,861 identifiers (67.0%)


/home/jwinchel/Heimdall/Heimdall/utils.py:457: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["gene_mapping:symbol_to_ensembl"] = gene_mapping.mapping_full


In [ ]:
gene_overlap = sorted(list(set(sc_data.var_names) & set(st_data.var_names)))

print(len(gene_overlap), "genes in common")

958 genes in common


In [26]:
import pandas as pd
pd.DataFrame(gene_overlap,columns=['ensembl_gene']).to_csv('/compute/oven-0-13/jwinchel/gene_overlap.csv',index=False)